## Error difference trend analysis

In this section the suspected trends concerning the statistics of error differences/changes will be investigated.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS, TWO_DAYS, FOUR_DAYS, PeriodicAggregation, ArbitraryAggregation, Aggregation
from python.temporal_hierarchy import TemporalHierarchy
from python.sampled_hierarchies import SampledHierarchies

In [ ]:
available_levels: list[Aggregation] = [
    TWO_DAYS,
    FOUR_DAYS,
    WEEKDAYS,
    WEEKENDS,
    WEEKS,
    FORTNIGHTS,
    ArbitraryAggregation('8Days1', [1] * 8 + [0] * 20),
    ArbitraryAggregation('8Days2', [0] * 8 + [1] * 8 + [0] * 12),
    ArbitraryAggregation('8Days3', [0] * 16 + [1] * 8 + [0] * 4),
    ArbitraryAggregation('16Days', [1] * 16 + [0] * 12),
    ArbitraryAggregation('12Days', [0] * 16 + [1] * 12),
    PeriodicAggregation('11L', 1, 1, True), # Skip every other
    PeriodicAggregation('11R', 1, 1, False),
    PeriodicAggregation('43L', 4, 3, True), # 4 Days on, 3 skip
    PeriodicAggregation('43R', 4, 3, False),
    PeriodicAggregation('34L', 3, 4, True), # 3 Days on, 4 skip
    PeriodicAggregation('34R', 3, 4, False),
    PeriodicAggregation('61L', 6, 1, True), # 6 Days on, 1 skip
    ArbitraryAggregation('AllWeekends', [0,0,0,0,0,1,1] * 4),
    ArbitraryAggregation('AllWeekdays', [1,1,1,1,1,0,0] * 4),
    ArbitraryAggregation('AllExtendedWeekends', [0,0,0,0,1,1,1] * 4),
    ArbitraryAggregation('AllExtendedWeekdays', [1,1,1,1,1,1,0] * 4),
    ArbitraryAggregation('10', [1, 0] * 14), # Zig zag
    ArbitraryAggregation('01', [0, 1] * 14)
]

In [ ]:
em = LoadedErrorMetrics('metrics/sample_hiers_errors.csv')

sh = SampledHierarchies(available_levels, 3, 10)

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy='Minimal'), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=h))
    for h in sh.hierarchies
])

First we calculate the statistics matrix as before, only difference being the error source, which are now the difference metrics. To calculate more meaningful mean and variance values, not only the top 0.1% of errors are sorted out, but also the bottom 0.1%, since when handling error changes, there can be extreme negative outliers as well.

In [ ]:
mon_diff_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=MONTHS, forecast_type=RECONCILED_FORECASTS),
                                       df_filter=lambda df: df[(df < np.quantile(df.values, 0.999)) & (df > np.quantile(df.values, 0.001))])
day_diff_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=DAYS, forecast_type=RECONCILED_FORECASTS),
                                       df_filter=lambda df: df[(df < np.quantile(df.values, 0.999)) & (df > np.quantile(df.values, 0.001))])

The same aggregation will happen, as last time, calculating several aggregates of the different statistical values across the hierarchies.

In [ ]:
mon_aggs = sh.aggregate_over(stats_matrix=mon_diff_stats_matrix,
                  group_by='heights',
                  stats=['mean', 'median', 'std'],
                  aggs=['mean', 'median', 'min', 'max'])

day_aggs = sh.aggregate_over(stats_matrix=day_diff_stats_matrix,
                  group_by='heights',
                  stats=['mean', 'median', 'std'],
                  aggs=['mean', 'median', 'min', 'max'])

### Monthly and daily error difference mean

The first error difference trend investigated will be the one, about the decreasing mean of error difference at both granularities, by having more hierarchy levels. I will start by investigating the monthly granularity error changes.

In [ ]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'mean'])

In [ ]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'median'])

Looking at either the mean or median aggregate of the mean statistics, it is apparent how the hypothesis was correct about the trend, there is a monoton negative change by adding more and more hierarchy levels.

In [ ]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'min'])

In [ ]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'max'])

The trend still holds after looking at the minimum and maximum aggregates, further proving the statement.

Next we look at daily granularity changes.

In [ ]:
plots.plot_aggregate_statistics(day_aggs['mean', 'mean'])

In [ ]:
plots.plot_aggregate_statistics(day_aggs['mean', 'median'])

When looking at the mean and median aggregate of the mean of daily error differences, the trend does seem to hold true to some extent, but it is not so consistent, and the range of change is very small as well.

In [ ]:
plots.plot_aggregate_statistics(day_aggs['mean', 'min'])

In [ ]:
plots.plot_aggregate_statistics(day_aggs['mean', 'max'])

Looking at the minimum and maximum aggregates, the trend does not seem to hold, further proving that the effects of this suspected trend is unsubstential.

Overall, the trend stating that having more hierarchy levels decreases the mean error change (meaning bigger improvements and lesser detriments) holds true for top level granularity, but not for bottom level granularity.

### Daily error difference median

The second hypothesis we look at, is about the worsening of daily granularity change median by adding more and more hierarchy levels.

In [ ]:
plots.plot_aggregate_statistics(day_aggs['median', 'mean'])

In [ ]:
plots.plot_aggregate_statistics(day_aggs['median', 'median'])

As we can see, when looking at the mean or median aggregates of the median statistic of daily error changes, there does seem to be a trend of increasing, which is consistent, but its range is very small.

In [ ]:
plots.plot_aggregate_statistics(day_aggs['median', 'min'])

In [ ]:
plots.plot_aggregate_statistics(day_aggs['median', 'max'])

When looking at the minimum and maximum aggregates, the trend is less consistent, but it still seems to more or less hold, still on the same small scale.

Let's look at a heatmap matrix visualising the median statistics per the different sampled hierarchies.

In [ ]:
plots.plot_sampled_error_matrix(sh, 'median', stats_matrix=day_diff_stats_matrix)

There were groups where the trend had more effect, with bigger increases, but there were also groups where the change was really small.

Overall, the trend stating that by having more aggregation levels, the median of error changes at bottom level granularity slightly increases, holds true, but on a very small scale.

### Conclusions

**Trends confirmed:**
- Monthly diff mean decreases with more levels — strong and consistent across all aggregates, including min/max.
- Daily diff median increases with more levels — consistent across mean/median aggregates, less so at min/max, but the range is very small.

**Trends not confirmed:**
- Daily diff mean does not show a meaningful trend — slightly visible at mean/median aggregates with very small range, falls apart at min/max.